Uncertainty Structure vs Magnitude


- **1A Shot-budget sweep**: $S \in \{10, 20, 50, 100, 200, 500, 1000, \infty\}$ with **K repetitions** and **mean ± 95% CI**
- **1B Noise-matched control**: inject Gaussian noise into deterministic (shots=∞) features with **matched per-feature variance**
- **1C Structure-destruction controls**: remove entanglement, or scramble phases (destroy coherence)


In [ ]:
# Install dependencies (Colab)
!pip -q install numpy pandas yfinance pennylane pennylane-lightning scikit-learn scipy matplotlib


In [ ]:
# Mount Google Drive and set output directory
from google.colab import drive
drive.mount('/content/drive')

import os
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
BASE_DIR = '/content/drive/MyDrive/Uncertainty_Resource/step1'
OUT_DIR = os.path.join(BASE_DIR, timestamp)
TABLE_DIR = os.path.join(OUT_DIR, 'tables')
FIG_DIR = os.path.join(OUT_DIR, 'figures')

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('Outputs will be saved to:', OUT_DIR)


In [ ]:
# Imports
import numpy as np
import pandas as pd
import yfinance as yf
import pennylane as qml

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

from scipy.stats import t

RNG = np.random.default_rng(123)


In [ ]:
# Metrics: ECE, Brier, mean±95% CI

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)


def brier_score(y_true, y_prob):
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob).astype(float)
    return float(np.mean((y_prob - y_true)**2))


def mean_ci95(values):
    v = np.asarray(values, dtype=float)
    n = v.size
    if n <= 1:
        return float(v.mean()), 0.0
    m = float(v.mean())
    s = float(v.std(ddof=1))
    se = s / np.sqrt(n)
    ci = float(t.ppf(0.975, df=n-1) * se)
    return m, ci


In [ ]:
# Data prep (financial forecasting with technical indicators)

class StockPredictor:
    def __init__(self, ticker='AAPL'):
        self.ticker = ticker
        self.scaler = StandardScaler()

    def _calculate_technical_indicators(self, df):
        df = df.copy()
        df['Returns'] = df['Close'].pct_change()

        for window in [5, 10, 20, 50]:
            df[f'SMA_{window}'] = df['Close'].rolling(window=window).mean()
            df[f'EMA_{window}'] = df['Close'].ewm(span=window, adjust=False).mean()

        df['Daily_Std'] = df['Returns'].rolling(window=20).std()
        df['Volume_SMA20'] = df['Volume'].rolling(window=20).mean()
        df['Volume_Change'] = df['Volume'].pct_change()
        df['ROC'] = df['Close'].pct_change(periods=10)

        return df.dropna()

    def prepare_data(self, start_date, end_date, feature_columns=None):
        stock = yf.Ticker(self.ticker)
        df = stock.history(start=start_date, end=end_date)
        if df.empty:
            raise ValueError(f'No data found for {self.ticker}')

        df = self._calculate_technical_indicators(df)

        if feature_columns is None:
            feature_columns = ['Returns', 'SMA_5', 'SMA_20', 'Daily_Std',
                              'Volume_SMA20', 'ROC', 'Volume_Change']

        X = df[feature_columns].values
        y = (df['Returns'].shift(-1) > 0).astype(int).values[:-1]
        X = X[:-1]

        X_scaled = self.scaler.fit_transform(X)
        return X_scaled, y


In [ ]:
# Quantum reservoir with configurable shots + structure controls

class QuantumReservoir:
    def __init__(self, n_qubits=6, connectivity=0.7, seed=0, shots=None, mode='full'):
        """
        shots: int or None (None => analytic expectation, i.e., shots=∞)
        mode: 'full' | 'no_entanglement' | 'phase_scramble'
        """
        self.n_qubits = int(n_qubits)
        self.connectivity = float(connectivity)
        self.seed = int(seed)
        self.shots = shots
        self.mode = mode

        self.dev = qml.device('default.qubit', wires=self.n_qubits, shots=self.shots)

        rng = np.random.default_rng(self.seed)
        # Simple random projection from input to qubit angles (kept from initial notebook idea)
        self.W = rng.normal(0, 1/np.sqrt(self.n_qubits), size=(self.n_qubits, self.n_qubits))
        mask = rng.random((self.n_qubits, self.n_qubits)) < self.connectivity
        self.W *= mask

        @qml.qnode(self.dev)
        def circuit(x, scramble=None):
            # Input encoding (angles)
            for i in range(self.n_qubits):
                qml.RY(x[i] * np.pi, wires=i)
                qml.RZ(x[i] * np.pi / 2, wires=i)

            # Optionally scramble phases (destroys coherent interference structure)
            if self.mode == 'phase_scramble' and scramble is not None:
                for i in range(self.n_qubits):
                    qml.RZ(scramble[i], wires=i)

            # Two layers (as in initial code)
            for _ in range(2):
                if self.mode != 'no_entanglement':
                    # all-to-all entanglement
                    for i in range(self.n_qubits):
                        for j in range(i + 1, self.n_qubits):
                            qml.CNOT(wires=[i, j])
                            qml.RZ(np.pi / 4, wires=j)

                # single-qubit mixing
                for i in range(self.n_qubits):
                    qml.Hadamard(wires=i)
                    qml.RY(np.pi / 2, wires=i)

            return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]

        self._circuit = circuit

    def featurize(self, X, rng=None):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError('X must be 2D array')
        if X.shape[1] < self.n_qubits:
            # pad if fewer features than qubits
            Xp = np.zeros((X.shape[0], self.n_qubits), dtype=float)
            Xp[:, :X.shape[1]] = X
            X = Xp
        else:
            X = X[:, :self.n_qubits]

        # project + tanh
        proj = np.tanh(X @ self.W.T)

        feats = []
        for i in range(proj.shape[0]):
            if self.mode == 'phase_scramble':
                if rng is None:
                    rng = np.random.default_rng(self.seed + 999)
                scramble = rng.uniform(0, 2*np.pi, size=self.n_qubits)
                feats.append(self._circuit(proj[i], scramble=scramble))
            else:
                feats.append(self._circuit(proj[i], scramble=None))
        return np.asarray(feats, dtype=float)


In [ ]:
# Train/evaluate helper

def run_readout(X_train_feat, y_train, X_test_feat, y_test, seed=0):
    clf = LogisticRegression(max_iter=2000, solver='lbfgs')
    clf.fit(X_train_feat, y_train)
    y_prob = clf.predict_proba(X_test_feat)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    out = {}
    out['accuracy'] = accuracy_score(y_test, y_pred)
    # ROC-AUC should use probabilities, not hard labels
    out['roc_auc'] = roc_auc_score(y_test, y_prob)
    out['nll'] = log_loss(y_test, np.vstack([1-y_prob, y_prob]).T, labels=[0,1])
    out['brier'] = brier_score(y_test, y_prob)
    out['ece'] = expected_calibration_error(y_test, y_prob, n_bins=10)
    return out


In [ ]:
# Step 1 experiment runner: shots sweep + noise-matched + structure-destruction controls

from tqdm.auto import tqdm

SHOTS_LIST = [10, 20, 50, 100, 200, 500, 1000, None]  # None => analytic (∞)
K_REPS = 10  # change to 20/30 for stronger CIs

TICKER = 'AAPL'
START_DATE = '2022-01-01'
END_DATE = '2024-01-01'
N_QUBITS = 6
CONNECTIVITY = 0.7

# Load and split data
sp = StockPredictor(TICKER)
X, y = sp.prepare_data(START_DATE, END_DATE)
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

conditions = [
    'quantum_full',
    'noise_matched',
    'no_entanglement',
    'phase_scramble',
]

rows = []

# Precompute deterministic (∞) features for noise-matched baseline (per rep we only add noise)
det_res = QuantumReservoir(n_qubits=N_QUBITS, connectivity=CONNECTIVITY, seed=0, shots=None, mode='full')
Xtr_det = det_res.featurize(X_train)
Xte_det = det_res.featurize(X_test)

for S in tqdm(SHOTS_LIST, desc='Shots sweep'):
    # For finite shots, estimate per-feature variance from repeated featurisation
    if S is None:
        var_diag = np.zeros(N_QUBITS)
    else:
        reps_feats = []
        for k in range(K_REPS):
            res_tmp = QuantumReservoir(n_qubits=N_QUBITS, connectivity=CONNECTIVITY, seed=1000+k, shots=S, mode='full')
            feats_k = res_tmp.featurize(X_train)
            reps_feats.append(feats_k)
        reps_feats = np.stack(reps_feats, axis=0)  # [K, N, D]
        # variance across repetitions, averaged over samples
        var_diag = reps_feats.var(axis=0, ddof=1).mean(axis=0)

    for k in range(K_REPS):
        base_seed = 10_000 + k
        rng = np.random.default_rng(base_seed)

        # --- quantum_full ---
        if S is None:
            # Analytic mode: deterministic
            Xtr = Xtr_det
            Xte = Xte_det
        else:
            res = QuantumReservoir(n_qubits=N_QUBITS, connectivity=CONNECTIVITY, seed=base_seed, shots=S, mode='full')
            Xtr = res.featurize(X_train)
            Xte = res.featurize(X_test)
        met = run_readout(Xtr, y_train, Xte, y_test, seed=base_seed)
        rows.append({
            'ticker': TICKER, 'start': START_DATE, 'end': END_DATE,
            'shots': (np.inf if S is None else S),
            'condition': 'quantum_full', 'rep': k, **met
        })

        # --- noise_matched (diagonal variance match) ---
        if S is None:
            # no noise when S=∞
            Xtr_nm, Xte_nm = Xtr_det, Xte_det
        else:
            noise_tr = rng.normal(0, np.sqrt(np.maximum(var_diag, 1e-12)), size=Xtr_det.shape)
            noise_te = rng.normal(0, np.sqrt(np.maximum(var_diag, 1e-12)), size=Xte_det.shape)
            Xtr_nm, Xte_nm = Xtr_det + noise_tr, Xte_det + noise_te
        met = run_readout(Xtr_nm, y_train, Xte_nm, y_test, seed=base_seed)
        rows.append({
            'ticker': TICKER, 'start': START_DATE, 'end': END_DATE,
            'shots': (np.inf if S is None else S),
            'condition': 'noise_matched', 'rep': k, **met
        })

        # --- no_entanglement ---
        if S is None:
            res = QuantumReservoir(n_qubits=N_QUBITS, connectivity=CONNECTIVITY, seed=base_seed, shots=None, mode='no_entanglement')
        else:
            res = QuantumReservoir(n_qubits=N_QUBITS, connectivity=CONNECTIVITY, seed=base_seed, shots=S, mode='no_entanglement')
        Xtr_ne = res.featurize(X_train)
        Xte_ne = res.featurize(X_test)
        met = run_readout(Xtr_ne, y_train, Xte_ne, y_test, seed=base_seed)
        rows.append({
            'ticker': TICKER, 'start': START_DATE, 'end': END_DATE,
            'shots': (np.inf if S is None else S),
            'condition': 'no_entanglement', 'rep': k, **met
        })

        # --- phase_scramble ---
        if S is None:
            res = QuantumReservoir(n_qubits=N_QUBITS, connectivity=CONNECTIVITY, seed=base_seed, shots=None, mode='phase_scramble')
        else:
            res = QuantumReservoir(n_qubits=N_QUBITS, connectivity=CONNECTIVITY, seed=base_seed, shots=S, mode='phase_scramble')
        Xtr_ps = res.featurize(X_train, rng=rng)
        Xte_ps = res.featurize(X_test, rng=rng)
        met = run_readout(Xtr_ps, y_train, Xte_ps, y_test, seed=base_seed)
        rows.append({
            'ticker': TICKER, 'start': START_DATE, 'end': END_DATE,
            'shots': (np.inf if S is None else S),
            'condition': 'phase_scramble', 'rep': k, **met
        })

results_df = pd.DataFrame(rows)
results_path = os.path.join(TABLE_DIR, 'results_step1_all.csv')
results_df.to_csv(results_path, index=False)
print('Saved:', results_path)
results_df.head()


In [ ]:
# Aggregate: mean ± 95% CI per shot and condition

summary_rows = []
for (shots, cond), g in results_df.groupby(['shots', 'condition']):
    for metric in ['accuracy', 'roc_auc', 'nll', 'brier', 'ece']:
        m, ci = mean_ci95(g[metric].values)
        summary_rows.append({
            'shots': shots,
            'condition': cond,
            'metric': metric,
            'mean': m,
            'ci95': ci,
            'n': len(g)
        })

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(TABLE_DIR, 'summary_ci95.csv')
summary_df.to_csv(summary_path, index=False)
print('Saved:', summary_path)
summary_df.head(10)


In [ ]:
# Visualisation: shot curves with CI

metric_titles = {
    'accuracy': 'Accuracy (↑)',
    'roc_auc': 'ROC-AUC (↑)',
    'nll': 'NLL (↓)',
    'brier': 'Brier (↓)',
    'ece': 'ECE (↓)'
}

plot_metrics = ['accuracy', 'roc_auc', 'ece', 'nll']
conditions_plot = ['quantum_full', 'noise_matched', 'no_entanglement', 'phase_scramble']

# For plotting, replace inf with a large number for x-axis, but label separately
shots_sorted = sorted(summary_df['shots'].unique(), key=lambda x: (np.isinf(x), x))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, metric in zip(axes, plot_metrics):
    for cond in conditions_plot:
        sub = summary_df[(summary_df['metric'] == metric) & (summary_df['condition'] == cond)].copy()
        sub = sub.sort_values('shots', key=lambda s: s.map(lambda x: (np.isinf(x), x)))
        x = sub['shots'].values
        y = sub['mean'].values
        yerr = sub['ci95'].values
        ax.errorbar(x, y, yerr=yerr, marker='o', capsize=3, linewidth=1, label=cond)

    ax.set_xscale('symlog', linthresh=50)
    ax.set_xlabel('Shots (symlog; ∞ shown at far right)')
    ax.set_ylabel(metric_titles[metric])
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle('Step 1: Shot-budget sweep with noise-matched + structure controls (mean ± 95% CI)')
fig.tight_layout(rect=[0, 0, 1, 0.96])

fig_path = os.path.join(FIG_DIR, 'shot_sweep_metrics_ci.png')
fig.savefig(fig_path, dpi=200)
print('Saved:', fig_path)
plt.show()


In [ ]:
# Summary plot: best-shot comparison (by mean ECE, tie-break by accuracy)

# Find best shot for quantum_full by lowest mean ECE (then highest accuracy)
q_ece = summary_df[(summary_df['condition']=='quantum_full') & (summary_df['metric']=='ece')].copy()
q_acc = summary_df[(summary_df['condition']=='quantum_full') & (summary_df['metric']=='accuracy')][['shots','mean']].rename(columns={'mean':'acc_mean'})
q = q_ece.merge(q_acc, on='shots', how='left')
q = q.sort_values(['mean','acc_mean'], ascending=[True, False])
BEST_SHOTS = float(q.iloc[0]['shots'])
print('Best shots (quantum_full) by ECE:', BEST_SHOTS)

metrics_show = ['accuracy','roc_auc','ece','nll','brier']
sub = summary_df[summary_df['shots']==BEST_SHOTS].copy()

# Pivot for plotting
pivot = sub.pivot_table(index='condition', columns='metric', values='mean')

fig, ax = plt.subplots(figsize=(12, 4))
# plot accuracy and ece only for clarity
for metric in ['accuracy','ece','nll']:
    pivot[metric].plot(kind='bar', ax=ax, alpha=0.9)

ax.set_title(f'Comparison at best shot (S={BEST_SHOTS})')
ax.set_ylabel('Metric value')
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()

fig_path = os.path.join(FIG_DIR, 'best_shot_comparison.png')
fig.savefig(fig_path, dpi=200)
print('Saved:', fig_path)
plt.show()


## Outputs
- All run-level metrics: `tables/results_step1_all.csv`
- Mean ± 95% CI summaries: `tables/summary_ci95.csv`
- Figures: `figures/*.png`

**Output folder:** printed near the top after Drive mount.